### llm judgement

In [ ]:
import re
import requests
import json
import pandas as pd
from openai import OpenAI

In [ ]:
from google.colab import userdata
proxy_model = userdata.get('PROXY_MODEL')
proxy_api_key = userdata.get('PROXY_API_KEY')
proxy_base_url = userdata.get('PROXY_BASE_URL')

In [ ]:
def clean_thought(text, expected_count):
    # 1. Удаляем блок размышлений
    text_ = re.sub(r'<thought>.*?</thought>', '', text, flags=re.DOTALL).strip()

    # 2. Паттерн для поиска
    pattern = r"score:\s*(\d+)\s*\|\s*llm_comment:\s*(.*?)(?=\s*score:|$)"
    matches = re.findall(pattern, text_, flags=re.DOTALL)

    # 3. Динамическая проверка
    if len(matches) != expected_count:
        raise ValueError(f"Мисматч ответов: ожидалось {expected_count}, получено {len(matches)}")

    return matches

In [ ]:
def build_batch_prompt(sub_df):
    query_id = []
    c_id = []
    prompts = []

    for idx, row in sub_df.iterrows():
        query_id.append(row['query_id'])
        c_id.append(row['candidate_id'])
        case_text = (
            f"КЕЙС №{idx+1}\n"
            f"Запрос рекрутера: {row['query_text']}\n"
            f"Краткий профиль кандидата: {row['full_resume']}\n"
            "---"
        )
        prompts.append(case_text)

    return query_id, c_id, prompts

In [ ]:
def get_reply(prompt, client):
  completion = client.chat.completions.create(
    model=proxy_model,
    messages=[
      {"role": "system", "content": """Ты — экспертный рекрутер. Ниже представлен список независимых кейсов (запрос + резюме)\n
        - Оценка 2: Полное совпадение или перекрытие стека, опыта.\n
        - Оценка 1: Частичное совпадение (есть база, но нет пары навыков).\n
        - Оценка 0: Не тот домен или нет ключевых навыков.\n
        - Опыт больше требуемого — это не плохо
        Для КАЖДОГО кейса по отдельности выведи результат в формате:\n
        score: [0, 1 или 2] | llm_comment: [твое краткое обоснование]\n
        Соблюдай строгую очередность. Не объединяй ответы.\n\n"""},
      {"role": "user", "content": prompt}
    ]
  )
  return completion.choices[0].message.content

fill table

In [ ]:
import os
from tqdm.auto import tqdm
import time

In [ ]:
def main(df, client):
  output_file = 'judgments.csv'
  batch_size = 5
  all_batch_results = []
  write_header = not os.path.exists(output_file)

  for i in tqdm(range(0, len(df), batch_size)):
      batch = df.iloc[i : i + batch_size]
      q_id, c_id, prompts = build_batch_prompt(batch)

      try:
            # Формируем единый промпт для батча
            full_prompt = '\n\n'.join(prompts)

            # Запрос к LLM с ретраями (упрощенно)
            reply = get_reply(full_prompt, client)
            clean_reply = clean_thought(reply, expected_count=len(batch))

            for (s, com), q, c, pr in zip(clean_reply, q_id, c_id, prompts):
                all_batch_results.append({
                    'query_id': q,
                    'candidate_id': c,
                    'relevance_score': int(s),
                    'judge_model': 'alibaba-qwen3-30b-a3b',
                    'judge_prompt_version': pr,
                    'judge_comment': com,
                })

            time.sleep(1) # Rate limit

      except Exception as e:
          print(f"\n❌ Ошибка на батче {i}: {e}")
          # Логирование упавшего батча в отдельный файл, чтобы не потерять данные
          with open("error_log.txt", "a") as f:
              f.write(f"Batch {i}: {str(e)}\n")
          continue # Пропускаем батч или делаем break, в зависимости от стратегии

      # Запись в файл чанками (например, каждые 5 батчей), чтобы не терять прогресс
      if len(all_batch_results) >= batch_size * 5:
          temp_df = pd.DataFrame(all_batch_results)
          temp_df.to_csv(output_file, mode='a', index=False, header=write_header)
          write_header = False # После первой записи заголовок больше не нужен
          all_batch_results = [] # Очистка буфера

  # Дозапись остатка
  if all_batch_results:
      temp_df = pd.DataFrame(all_batch_results)
      temp_df.to_csv(output_file, mode='a', index=False, header=write_header)

In [ ]:
client = OpenAI(api_key=proxy_api_key,
base_url=proxy_base_url)

In [ ]:
main(pooled_candidates_to_judge_res, client)

  0%|          | 0/351 [00:00<?, ?it/s]